In [5]:
# Cell 1: Import required deep learning and visualization libraries
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import datasets
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Sequential

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.22.0-rc0


In [6]:
# Cell 2: Load a 500-sample subset of CIFAR-10 for fast CPU training
(x_train_raw, y_train_raw), _ = datasets.cifar10.load_data()

# 1. Take 500 real images (keeps CPU training time under 45 seconds)
num_samples = 500
x_subset = x_train_raw[:num_samples]
y_train = y_train_raw[:num_samples].flatten()

# 2. Resize from 32x32 to 128x128 (compatible with MobileNetV2) and normalize to [0, 1]
print("Preprocessing and resizing images to 128x128...")
X_train = tf.image.resize(x_subset, (128, 128)).numpy().astype(np.float32) / 255.0

print(f"Data ready: X_train shape = {X_train.shape}, y_train shape = {y_train.shape}")

Preprocessing and resizing images to 128x128...
Data ready: X_train shape = (500, 128, 128, 3), y_train shape = (500,)


In [7]:
# Cell 3: Step 1 (Load Model), Step 2 (Freeze Layers), Step 3 (Add Classifier)

# Step 1: Load pre-trained MobileNetV2 without the top ImageNet classification layer
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(128, 128, 3)
)

# Step 2: Freeze all lower convolutional layers
base_model.trainable = False

# Step 3: Add custom dense classification layers on top
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dense(10, activation='softmax')
])

# Compile model for initial training
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_128            │ (None, 4, 4, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,423,242 (9.24 MB)

 Trainable params: 165,258 (645.54 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [8]:
# Cell 4: Step 4 (Train Classifier) and Step 5 (Fine-Tuning)

# Step 4: Train only the top classifier layers on the dataset (base frozen)
print("--- Phase 1: Training Classifier Head (Base Frozen) ---")
history_initial = model.fit(
    X_train, y_train,
    epochs=3,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# Step 5: Unfreeze the top 20 convolutional layers for fine-tuning
print("\n--- Phase 2: Fine-Tuning Top 20 Layers ---")
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

# Recompile with a low learning rate to protect pre-trained weights
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_finetune = model.fit(
    X_train, y_train,
    epochs=3,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)
print("Training and fine-tuning complete!")

--- Phase 1: Training Classifier Head (Base Frozen) ---
Epoch 1/3
13/13 ━━━━━━━━━━━━━━━━━━━━ 14s 817ms/step - accuracy: 0.3625 - loss: 1.9769 - val_accuracy: 0.4700 - val_loss: 1.4665
Epoch 2/3
13/13 ━━━━━━━━━━━━━━━━━━━━ 10s 307ms/step - accuracy: 0.7075 - loss: 0.9335 - val_accuracy: 0.6400 - val_loss: 1.1009
Epoch 3/3
13/13 ━━━━━━━━━━━━━━━━━━━━ 5s 318ms/step - accuracy: 0.8050 - loss: 0.6238 - val_accuracy: 0.6600 - val_loss: 0.9502

--- Phase 2: Fine-Tuning Top 20 Layers ---
Epoch 1/3
13/13 ━━━━━━━━━━━━━━━━━━━━ 16s 824ms/step - accuracy: 0.5950 - loss: 1.2123 - val_accuracy: 0.6800 - val_loss: 0.9307
Epoch 2/3
13/13 ━━━━━━━━━━━━━━━━━━━━ 9s 366ms/step - accuracy: 0.6925 - loss: 1.0158 - val_accuracy: 0.6500 - val_loss: 0.9239
Epoch 3/3
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 299ms/step - accuracy: 0.7475 - loss: 0.8778 - val_accuracy: 0.6400 - val_loss: 0.9233
Training and fine-tuning complete!
